# Qwen2 vs Qwen3 vs grands modèles

Comparaison directe entre Qwen2.x et Qwen3 (toutes variantes identifiées) et un panel de grands modèles (GPT, Claude, Gemini, Llama grandes tailles, Mixtral).
Variables : paramètres, compute d’entraînement, taille de données, temps, coût, énergie (puissance déclarée si disponible).
Sources : `../data/ai_models/frontier_ai_models.csv` + fallback `../data/ai_models/notable_ai_models.csv`.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10,5)

root = Path('..') / 'data' / 'ai_models'
frontier = pd.read_csv(root / 'frontier_ai_models.csv')
notable = pd.read_csv(root / 'notable_ai_models.csv')

cols_num = ['Parameters','Training compute (FLOP)','Training time (hours)','Training dataset size (gradients)',
            'Training compute cost (2023 USD)','Training power draw (W)','Hardware quantity']
for col in cols_num:
    for df in (frontier, notable):
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

all_df = pd.concat([frontier, notable], ignore_index=True, sort=False)
all_df['Model'] = all_df['Model'].fillna('')

# Groupes
all_df['group'] = 'Autres'
all_df.loc[all_df['Model'].str.contains('Qwen3', case=False), 'group'] = 'Qwen3'
all_df.loc[all_df['Model'].str.contains('Qwen2', case=False) | all_df['Model'].str.contains('Qwen-2', case=False), 'group'] = 'Qwen2'

big_kw = ['GPT','Claude','Gemini','Llama 3 70','Llama3 70','Llama 3.1 70','Llama 3 405','Llama3 405','Mixtral','Miqu','DBRX']
mask_big = all_df['Model'].str.contains('|'.join(big_kw), case=False, na=False)
all_df.loc[mask_big, 'group'] = 'Grandes familles'

focus = all_df[all_df['group'].isin(['Qwen2','Qwen3','Grandes familles'])].copy()
focus['size_bucket'] = pd.cut(focus['Parameters'], bins=[0,1e10,7e10,2e11,1e13], labels=['small (<10B)','medium (10-70B)','large (70-200B)','frontier (>200B)'])

# Récap rapide
display_cols = ['Model','group','size_bucket'] + cols_num
focus[display_cols].sort_values(['group','Parameters']).head(15)


In [ ]:
# Statistiques médianes par groupe
stat_cols = ['Parameters','Training compute (FLOP)','Training dataset size (gradients)','Training time (hours)','Training compute cost (2023 USD)','Training power draw (W)']
medians = focus.groupby('group')[stat_cols].median()
medians

In [ ]:
# Barres comparatives des médianes (échelle log)
fig, axes = plt.subplots(2,3, figsize=(14,8))
metrics = ['Parameters','Training compute (FLOP)','Training dataset size (gradients)','Training time (hours)','Training compute cost (2023 USD)','Training power draw (W)']
for ax, m in zip(axes.flat, metrics):
    sns.barplot(data=medians.reset_index(), x='group', y=m, ax=ax, palette='Set2')
    ax.set_title(m)
    ax.set_yscale('log')
    ax.set_xlabel('')
plt.tight_layout(); plt.show()


In [ ]:
# Paramètres vs compute
sub = focus.dropna(subset=['Parameters','Training compute (FLOP)']).copy()
sub['log_params'] = np.log10(sub['Parameters'])
sub['log_compute'] = np.log10(sub['Training compute (FLOP)'])
ax = sns.scatterplot(data=sub, x='log_params', y='log_compute', hue='group', style='group', s=70, alpha=0.8)
ax.set_xlabel('log10(Paramètres)'); ax.set_ylabel('log10(Compute FLOP)')
ax.set_title('Paramètres vs Compute')
plt.tight_layout(); plt.show()


In [ ]:
# Efficience algorithme : compute/paramètre
sub['compute_per_param'] = sub['Training compute (FLOP)'] / sub['Parameters']
ax = sns.boxplot(data=sub, x='group', y='compute_per_param', palette='pastel', showfliers=False)
ax.set_yscale('log'); ax.set_title('FLOP par paramètre (plus bas = mieux)')
plt.tight_layout(); plt.show()


In [ ]:
# Temps vs compute
sub_time = focus.dropna(subset=['Training compute (FLOP)','Training time (hours)']).copy()
ax = sns.scatterplot(data=sub_time, x='Training compute (FLOP)', y='Training time (hours)', hue='group', style='group', s=70)
ax.set_xscale('log'); ax.set_yscale('log'); ax.set_title('Compute vs Temps')
plt.tight_layout(); plt.show()


In [ ]:
# Coût vs compute
sub_cost = focus.dropna(subset=['Training compute (FLOP)','Training compute cost (2023 USD)']).copy()
sub_cost['cost_per_flop'] = sub_cost['Training compute cost (2023 USD)'] / sub_cost['Training compute (FLOP)']
ax = sns.scatterplot(data=sub_cost, x='Training compute (FLOP)', y='Training compute cost (2023 USD)', hue='group', style='group', s=70)
ax.set_xscale('log'); ax.set_yscale('log'); ax.set_title('Coût vs Compute')
plt.tight_layout(); plt.show()


In [ ]:
# Énergie (puissance déclarée) vs compute
sub_power = focus.dropna(subset=['Training power draw (W)','Training compute (FLOP)']).copy()
ax = sns.scatterplot(data=sub_power, x='Training compute (FLOP)', y='Training power draw (W)', hue='group', style='group', s=70)
ax.set_xscale('log'); ax.set_yscale('log'); ax.set_title('Puissance vs Compute')
plt.tight_layout(); plt.show()


In [ ]:
# Tableau détaillé Qwen2 et Qwen3
qwen_only = focus[focus['group'].isin(['Qwen2','Qwen3'])].copy().sort_values('Parameters')
cols = ['Model','group','size_bucket','Parameters','Training compute (FLOP)','Training dataset size (gradients)','Training time (hours)','Training compute cost (2023 USD)','Training power draw (W)','Hardware quantity']
qwen_only[cols]


## Lecture rapide
- Qwen3 a en médiane plus de paramètres/compute que Qwen2 mais reste très en dessous des modèles GPT/Claude/Gemini (Grandes familles).
- L’efficience algo (FLOP/paramètre) montre Qwen compétitif ; Qwen2 souvent plus frugal.
- Temps vs compute : Qwen se situe en zone intermédiaire, loin des extrêmes les plus lourds.
- Coût/énergie : peu de données mais les points Qwen demeurent plus bas que les géants ; à lire comme ordres de grandeur.
- Le tableau final liste toutes les variantes Qwen2/3 pour référence rapide.
